In [5]:
import nibabel as nib
from pathlib import Path
import numpy as np
import itertools
from sklearn.metrics import jaccard_score, adjusted_rand_score, f1_score
import pandas as pd
from tqdm import trange
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'Arial'

DATA_PATH = Path('../data')
RESULTS_PATH = Path('../results')
PLOTS_PATH = Path('../plots')
NII_DATA_PATH = RESULTS_PATH / 'nii'
PLOT_KWARGS_DICT = dict(bbox_inches='tight', transparent=True, dpi=600)
NAME_MAPPING_DICT = {
    'BCP (0-3)': 'BCP_age_0_3_resting_remap',
    'HCPD (05-11)': 'HCP_D_age_05_11_resting',
    'HCPD (12-21)': 'HCP_D_age_12_21_resting',
    'HCPYA (22-35)': 'HCP_YA_age_22_35_resting',
    'HCPA (36-60)': 'HCP_A_age_36_60_resting',
    'HCPA (61-100)': 'HCP_A_age_61_100_resting'
}

In [6]:
def max_jaccard_score(y_true, y_pred, n):
    numbers = list(range(1, n + 1))
    scores = []
    for perm in itertools.permutations(numbers):
        correspondence = dict(zip(numbers, perm))
        new_y_pred = np.vectorize(correspondence.get)(y_pred)
        score = jaccard_score(y_true=y_true, y_pred=new_y_pred, average='macro')
        scores.append(score)
    return max(scores)


def max_dice_score(y_true, y_pred, n):
    numbers = list(range(1, n + 1))
    scores = []
    for perm in itertools.permutations(numbers):
        correspondence = dict(zip(numbers, perm))
        new_y_pred = np.vectorize(correspondence.get)(y_pred)
        score = f1_score(y_true, new_y_pred, average='macro')
        scores.append(score)
    return max(scores)


def set_upper_tri_nan(df):
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            if i < j:
                df.iloc[i, j] = np.nan
    return df

# Connectivity Based

In [10]:
for cluster_n in trange(2, 7):
    file_name_mapping = {k: DATA_PATH / f'cbptools/{v}/K{cluster_n}.nii.gz' for (k, v) in NAME_MAPPING_DICT.items()}
    result = []
    for name_1, name_2 in itertools.combinations(file_name_mapping.keys(), 2):
        file_1 = file_name_mapping[name_1]
        file_2 = file_name_mapping[name_2]

        print(name_1, name_2)
        data_1 = nib.load(file_1).get_fdata()
        data_2 = nib.load(file_2).get_fdata()

        mask_1 = data_1 > 0
        mask_2 = data_2 > 0
        shared_mask = mask_1 & mask_2

        data_1 = np.round(data_1)[shared_mask]
        data_2 = np.round(data_2)[shared_mask]

        score = max_dice_score(data_1, data_2, cluster_n)
        result.append(dict(name_1=name_1, name_2=name_2, score=score))
        result.append(dict(name_1=name_2, name_2=name_1, score=score))
    result = pd.DataFrame(result)
    result = result.pivot(index='name_1', columns='name_2', values='score')
    result = result.loc[list(NAME_MAPPING_DICT.keys()), list(NAME_MAPPING_DICT.keys())]
    result.to_csv(RESULTS_PATH / f'scores/Dice_K{cluster_n}.csv')

  0%|          | 0/5 [00:00<?, ?it/s]

BCP (0-3) HCPD (05-11)


ValueError: operands could not be broadcast together with shapes (97,116,79) (91,109,91) 

In [ ]:
for cluster_n in range(2, 7):
    result = pd.read_csv(RESULTS_PATH / f'scores/Dice_K{cluster_n}.csv', index_col=0)
    result = result.iloc[1:, :-1]
    result = set_upper_tri_nan(result)
    print(f'K = {cluster_n}, Mean Dice = {np.nanmean(result):.3f} SD Dice = {np.nanstd(result, ddof=0):.3f}')
    fig, ax = plt.subplots(figsize=(3, 3))
    sns.heatmap(result, annot=True, fmt='.2f', vmin=0.5, vmax=0.85,
                ax=ax, cbar=False, cmap='YlOrBr')
    ax.set_ylabel('')
    ax.set_xlabel('')
    fig.savefig(PLOTS_PATH / f'heatmap/Dice_{cluster_n}.svg', **PLOT_KWARGS_DICT)